### Step 1 - Read csv file

In [0]:
%run ../00-common/01.environment_config

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
sources_path = f"{landing_folder_path}/{v_batch_id}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,DateType

In [0]:
schema=StructType([
    StructField('date',DateType()),
    StructField('raceName',StringType()),
    StructField('round',IntegerType()),
    StructField('season',IntegerType()),
    StructField('url',StringType()),
    StructField('constructorId',StringType()),
    StructField('driverId',StringType()),
    StructField('grid',IntegerType()),
    StructField('laps',IntegerType()),
    StructField('number',IntegerType()),
    StructField('points',DoubleType()),
    StructField('position',IntegerType()),
    StructField('positionText',StringType()),
    StructField('status',StringType())

    ]
)

In [0]:
results_df=(
    spark.read.format('json')
    #.option('header',True)
    #.option('inferSchema',True)
    .option('mode','FAILFAST')
    
    .schema(schema)
    .load(sources_path))

### Step 2 - Ingest time stamp and file path

In [0]:
from pyspark.sql.functions import *

In [0]:
results_final_df = add_ingestion_metadata(results_df)

### Step3- Write data to Bronze delta table

In [0]:
write_data_to_bronze(results_final_df,table_name, v_batch_id)

In [0]:
display(spark.table(table_name))